# 02 Parse Bronze Tables

Read downloaded ZIPs from the raw manifest, parse MMSDM CSV content, append Bronze Delta tables, write file audit rows, and quarantine malformed files.

## Configure Bronze Parse Run

This cell creates a run ID, limits the number of raw ZIP files parsed in one run, and defines the quarantine path for bad files.

In [ ]:
# Cell purpose: Configure Bronze Parse Run.
from datetime import datetime, timezone
from pathlib import Path
import uuid

run_id = str(uuid.uuid4())
max_files_per_run = 500
quarantine_path = "Files/nemweb/quarantine"

print(f"run_id={run_id}")

## Bootstrap Local Project Package

This cell makes the uploaded `nem_fabric` source package importable in Fabric. Upload `src/nem_fabric` to `Files/libs/nem_fabric` before running the notebook in a Pipeline.

In [ ]:
# Cell purpose: Make nem_fabric importable from Lakehouse Files.
import os
import sys

fabric_lib_path = os.getenv("FABRIC_NOTEBOOK_LIB_PATH", "/lakehouse/default/Files/libs")
if fabric_lib_path not in sys.path:
    sys.path.insert(0, fabric_lib_path)

print(f"Python library path ready: {fabric_lib_path}")

## Import Parser and Define Helpers

This cell imports the MMSDM parser and defines Fabric helper functions for table checks and binary reads from Lakehouse Files.

In [ ]:
# Cell purpose: Import Parser and Define Helpers.
from nem_fabric.mmsdm_parser import parse_zip_bytes


def table_exists(table_name: str) -> bool:
    """Return True when a Lakehouse table exists in the current Spark catalogue."""
    return spark.catalog.tableExists(table_name)


def read_lakehouse_binary(relative_path: str) -> bytes:
    """Read ZIP bytes from the default Lakehouse Files mount."""
    mounted_path = Path("/lakehouse/default") / relative_path
    with mounted_path.open("rb") as file:
        return file.read()


if not table_exists("nem_raw_zip_manifest"):
    raise RuntimeError("nem_raw_zip_manifest does not exist. Run notebook 01 first.")

## Select Unparsed ZIP Files

This cell reads the raw ZIP manifest and removes files already present in the file audit table, making Bronze parsing idempotent.

In [ ]:
# Cell purpose: Select Unparsed ZIP Files.
manifest = spark.table("nem_raw_zip_manifest").filter("status = 'downloaded'")
if table_exists("nem_raw_file_audit"):
    parsed = spark.table("nem_raw_file_audit").select("source_url").distinct()
    manifest = manifest.join(parsed, on="source_url", how="left_anti")

files_to_parse = manifest.orderBy("file_datetime").limit(max_files_per_run).collect()
print(f"Files selected for Bronze parsing: {len(files_to_parse)}")

## Parse ZIP Files and Build Audit Records

This cell parses each selected ZIP, preserves row-level metadata, collects Bronze DataFrames, and records parsing outcomes for auditability.

In [ ]:
# Cell purpose: Parse ZIP Files and Build Audit Records.
bronze_frames = []
audit_rows = []

for item in files_to_parse:
    parsed_at = datetime.now(timezone.utc).isoformat()
    status = "parsed"
    error_message = ""
    table_count = 0
    row_count = 0
    try:
        zip_bytes = read_lakehouse_binary(item.lakehouse_path)
        tables = parse_zip_bytes(zip_bytes, item.source_url)
        table_count = len(tables)
        for table in tables:
            pdf = table.dataframe.copy()
            pdf["run_id"] = run_id
            pdf["source_name"] = item.source_name
            pdf["bronze_loaded_datetime"] = parsed_at
            row_count += len(pdf)
            if not pdf.empty:
                bronze_frames.append(pdf)
    except Exception as exc:
        status = "failed"
        error_message = str(exc)[:4000]
        try:
            source_file = Path("/lakehouse/default") / item.lakehouse_path
            target_file = Path("/lakehouse/default") / quarantine_path / item.source_zip_name
            target_file.parent.mkdir(parents=True, exist_ok=True)
            target_file.write_bytes(source_file.read_bytes())
        except Exception as quarantine_exc:
            error_message = f"{error_message}; quarantine_failed={quarantine_exc}"[:4000]

    audit_rows.append({
        "run_id": run_id,
        "source_name": item.source_name,
        "source_url": item.source_url,
        "source_zip_name": item.source_zip_name,
        "lakehouse_path": item.lakehouse_path,
        "parsed_datetime": parsed_at,
        "status": status,
        "table_count": table_count,
        "row_count_bronze": row_count,
        "error_message": error_message,
    })

## Write Bronze and Audit Tables

This cell appends parsed MMSDM rows to Bronze Delta tables, creates source-specific Bronze subsets, and writes file audit results.

In [ ]:
# Cell purpose: Write Bronze and Audit Tables.
import pandas as pd

if bronze_frames:
    bronze_pdf = pd.concat(bronze_frames, ignore_index=True, sort=False).fillna("")
    bronze_sdf = spark.createDataFrame(bronze_pdf.astype(str))
    bronze_sdf.write.format("delta").mode("append").saveAsTable("nem_bronze_mmsdm_rows")

    for table_name, filter_expr in {
        "nem_bronze_dispatchis": "package_name = 'DISPATCH'",
        "nem_bronze_public_prices": "upper(table_name) like '%PRICE%'",
        "nem_bronze_interconnector": "upper(table_name) like '%INTERCONNECT%'",
        "nem_bronze_generation": "upper(table_name) like '%GEN%' or upper(table_name) like '%SCADA%'",
    }.items():
        subset = bronze_sdf.filter(filter_expr)
        if subset.limit(1).count() > 0:
            subset.write.format("delta").mode("append").saveAsTable(table_name)
else:
    print("No Bronze rows produced.")

if audit_rows:
    spark.createDataFrame(audit_rows).write.format("delta").mode("append").saveAsTable("nem_raw_file_audit")
    display(spark.createDataFrame(audit_rows))
else:
    print("No files required parsing.")